# HFENN Bias Drift Kalman 实验

## 目标
在 Online-causal（可部署）设定下，用 Kalman 滤波器估计随时间漂移的偏置 $b_t$

## 评估线
1. Online-causal + Base
2. Online-causal + Base + Kalman
3. Online-causal + Fine-tune
4. Online-causal + Fine-tune + Kalman
5. Online-causal + Base + AR(1)

In [ ]:
# === Cell 1: 导入依赖 ===
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import pearsonr
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import warnings
warnings.filterwarnings('ignore')
print("依赖导入完成")

In [ ]:
# === Cell 2: Bias Drift Kalman 滤波器类 ===
class BiasDriftKalman:
    def __init__(self, Q, R, b0=0.0, P0=1.0):
        self.Q = Q
        self.R = R
        self.b = b0
        self.P = P0
    def predict(self):
        return self.b, self.P + self.Q
    def update(self, r_t):
        b_pred, P_pred = self.predict()
        K = P_pred / (P_pred + self.R)
        self.b = b_pred + K * (r_t - b_pred)
        self.P = (1 - K) * P_pred
        return self.b
    def filter_sequence(self, residuals):
        b_estimates = np.zeros(len(residuals))
        for t in range(len(residuals)):
            b_estimates[t] = self.update(residuals[t])
        return b_estimates
    def get_state(self):
        return self.b, self.P
    def set_state(self, b, P):
        self.b = b
        self.P = P

class AR1Corrector:
    def __init__(self, phi=None):
        self.phi = phi
        self.last_residual = 0.0
    def fit(self, residuals):
        if len(residuals) < 2:
            self.phi = 0.0
            return
        self.phi = np.corrcoef(residuals[:-1], residuals[1:])[0, 1]
        if np.isnan(self.phi):
            self.phi = 0.0
    def correct_sequence(self, y_pred, y_true):
        y_corrected = np.zeros(len(y_pred))
        for t in range(len(y_pred)):
            y_corrected[t] = y_pred[t] + self.phi * self.last_residual
            self.last_residual = y_true[t] - y_pred[t]
        return y_corrected

print("✅ BiasDriftKalman 和 AR1Corrector 类定义完成")

In [ ]:
# === Cell 3: Q/R 网格搜索 ===
def compute_initial_QR(residuals):
    R0 = np.var(residuals)
    Q0 = np.var(np.diff(residuals))
    return Q0, R0

def grid_search_QR(y_true_calib, y_pred_calib, Q_mults=[0.25,0.5,1,2,4,8], R_mults=[0.25,0.5,1,2,4], val_ratio=0.2):
    n_calib = len(y_true_calib)
    n_train = int(n_calib * (1 - val_ratio))
    y_true_train, y_pred_train = y_true_calib[:n_train], y_pred_calib[:n_train]
    y_true_val, y_pred_val = y_true_calib[n_train:], y_pred_calib[n_train:]
    residuals_train = y_true_train - y_pred_train
    Q0, R0 = compute_initial_QR(residuals_train)
    print(f"  初始化: Q0={Q0:.6f}, R0={R0:.6f}")
    results = []
    for q_m in Q_mults:
        for r_m in R_mults:
            Q, R = Q0 * q_m, R0 * r_m
            kf = BiasDriftKalman(Q=Q, R=R, b0=0.0, P0=R0)
            kf.filter_sequence(residuals_train)
            b_f, P_f = kf.get_state()
            kf_val = BiasDriftKalman(Q=Q, R=R)
            kf_val.set_state(b_f, P_f)
            b_val = kf_val.filter_sequence(y_true_val - y_pred_val)
            mae_val = mean_absolute_error(y_true_val, y_pred_val + b_val)
            results.append({'Q_mult':q_m, 'R_mult':r_m, 'Q':Q, 'R':R, 'MAE_val':mae_val})
    results_df = pd.DataFrame(results)
    best_idx = results_df['MAE_val'].idxmin()
    best_Q, best_R = results_df.loc[best_idx, 'Q'], results_df.loc[best_idx, 'R']
    print(f"  最优: Q={best_Q:.6f}, R={best_R:.6f}, MAE={results_df.loc[best_idx,'MAE_val']:.6f}")
    return best_Q, best_R, Q0, R0, results_df

print("✅ Q/R 网格搜索定义完成")

In [ ]:
# === Cell 4: 评估工具 ===
def compute_metrics(y_true, y_pred):
    return {'R2': r2_score(y_true, y_pred), 'MAE': mean_absolute_error(y_true, y_pred), 'RMSE': np.sqrt(mean_squared_error(y_true, y_pred))}

def compute_bucket_metrics(y_true, y_pred, n_buckets=5):
    quantiles = np.percentile(y_true, np.linspace(0, 100, n_buckets + 1))
    results = []
    for i in range(n_buckets):
        low, high = quantiles[i], quantiles[i + 1]
        mask = (y_true >= low) & (y_true <= high) if i == n_buckets - 1 else (y_true >= low) & (y_true < high)
        if mask.sum() > 0:
            results.append({'Bucket': i+1, 'Range': f'[{low:.3f},{high:.3f}]', 'N': mask.sum(), 'MAE': mean_absolute_error(y_true[mask], y_pred[mask])})
    return pd.DataFrame(results)

def block_bootstrap_test(y_true, y_pred_a, y_pred_b, n_bootstrap=1000, block_size=12):
    n = len(y_true)
    n_blocks = n // block_size
    delta_samples = []
    for _ in range(n_bootstrap):
        block_idx = np.random.choice(n_blocks, size=n_blocks, replace=True)
        sample_idx = np.array([j for bi in block_idx for j in range(bi*block_size, min((bi+1)*block_size, n))])[:n]
        r2_a = r2_score(y_true[sample_idx], y_pred_a[sample_idx])
        r2_b = r2_score(y_true[sample_idx], y_pred_b[sample_idx])
        delta_samples.append(r2_b - r2_a)
    delta_samples = np.array(delta_samples)
    return {'delta_r2_mean': np.mean(delta_samples), 'ci_low': np.percentile(delta_samples, 2.5), 'ci_high': np.percentile(delta_samples, 97.5), 'win_rate': np.mean(delta_samples > 0)}

def compute_acf(residuals, max_lag=10):
    acf = [1.0]
    for lag in range(1, max_lag + 1):
        if len(residuals) > lag:
            acf.append(np.corrcoef(residuals[:-lag], residuals[lag:])[0, 1])
        else:
            acf.append(np.nan)
    return np.array(acf)

def compute_run_stats(residuals):
    signs = np.sign(residuals)
    runs = []
    curr_sign, curr_len = signs[0], 1
    for i in range(1, len(signs)):
        if signs[i] == curr_sign:
            curr_len += 1
        else:
            runs.append(curr_len)
            curr_sign, curr_len = signs[i], 1
    runs.append(curr_len)
    return {'n_runs': len(runs), 'mean_run_length': np.mean(runs), 'max_run_length': max(runs)}

print("✅ 评估工具定义完成")

In [ ]:
# === Cell 5: 数据加载检查 ===
print("="*70)
print("数据加载检查")
print("="*70)
required_vars = ['y_all', 'y_pred_base_all', 'y_pred_ft_all', 'online_causal_calib_indices', 'online_causal_test_indices']
missing = [v for v in required_vars if v not in dir()]
if missing:
    print(f"⚠️ 缺少变量: {missing}")
    print("请先运行 HFENN_affine_calibration_exp.ipynb")
else:
    print("✅ 所有变量已加载")

In [ ]:
# === Cell 6: 准备数据 ===
print("="*70)
print("Step 1: 准备 Online-causal 数据")
print("="*70)
calib_idx = online_causal_calib_indices
test_idx = online_causal_test_indices
y_true_calib = y_all[calib_idx]
y_pred_base_calib = y_pred_base_all[calib_idx]
y_pred_ft_calib = y_pred_ft_all[calib_idx]
y_true_test = y_all[test_idx]
y_pred_base_test = y_pred_base_all[test_idx]
y_pred_ft_test = y_pred_ft_all[test_idx]
residuals_base_calib = y_true_calib - y_pred_base_calib
residuals_ft_calib = y_true_calib - y_pred_ft_calib
print(f"校准段: {len(calib_idx)} 窗口, 测试段: {len(test_idx)} 窗口")

In [ ]:
# === Cell 7: Base + Kalman 参数搜索 ===
print("="*70)
print("Step 2: Base + Kalman 参数搜索")
print("="*70)
best_Q_base, best_R_base, Q0_base, R0_base, grid_base = grid_search_QR(y_true_calib, y_pred_base_calib)

In [ ]:
# === Cell 8: Fine-tune + Kalman 参数搜索 ===
print("="*70)
print("Step 3: Fine-tune + Kalman 参数搜索")
print("="*70)
best_Q_ft, best_R_ft, Q0_ft, R0_ft, grid_ft = grid_search_QR(y_true_calib, y_pred_ft_calib)

In [ ]:
# === Cell 9: AR(1) 拟合 ===
print("="*70)
print("Step 4: AR(1) 参数拟合")
print("="*70)
ar1_base = AR1Corrector()
ar1_base.fit(residuals_base_calib)
print(f"Base AR(1) phi = {ar1_base.phi:.4f}")

In [ ]:
# === Cell 10: 应用各方法到测试段 ===
print("="*70)
print("Step 5: 应用各方法到测试段")
print("="*70)

# Base
y_pred_test_base = y_pred_base_test.copy()

# Base + Kalman (从校准段继续)
kf_base = BiasDriftKalman(Q=best_Q_base, R=best_R_base, b0=0.0, P0=R0_base)
kf_base.filter_sequence(residuals_base_calib)
b_f, P_f = kf_base.get_state()
print(f"Base Kalman 校准段结束: b={b_f:.6f}")
kf_base_test = BiasDriftKalman(Q=best_Q_base, R=best_R_base)
kf_base_test.set_state(b_f, P_f)
b_test_base = kf_base_test.filter_sequence(y_true_test - y_pred_base_test)
y_pred_test_base_kalman = y_pred_base_test + b_test_base

# Fine-tune
y_pred_test_ft = y_pred_ft_test.copy()

# Fine-tune + Kalman
kf_ft = BiasDriftKalman(Q=best_Q_ft, R=best_R_ft, b0=0.0, P0=R0_ft)
kf_ft.filter_sequence(residuals_ft_calib)
b_f_ft, P_f_ft = kf_ft.get_state()
print(f"FT Kalman 校准段结束: b={b_f_ft:.6f}")
kf_ft_test = BiasDriftKalman(Q=best_Q_ft, R=best_R_ft)
kf_ft_test.set_state(b_f_ft, P_f_ft)
b_test_ft = kf_ft_test.filter_sequence(y_true_test - y_pred_ft_test)
y_pred_test_ft_kalman = y_pred_ft_test + b_test_ft

# Base + AR(1)
ar1_test = AR1Corrector(phi=ar1_base.phi)
ar1_test.last_residual = residuals_base_calib[-1]
y_pred_test_base_ar1 = ar1_test.correct_sequence(y_pred_base_test, y_true_test)

print("✅ 所有方法已应用")

In [ ]:
# === Cell 11: 评估指标 ===
print("="*70)
print("Step 6: 评估各方法性能")
print("="*70)
methods = {'Base': y_pred_test_base, 'Base+Kalman': y_pred_test_base_kalman, 'Fine-tune': y_pred_test_ft, 'FT+Kalman': y_pred_test_ft_kalman, 'Base+AR(1)': y_pred_test_base_ar1}
metrics_results = [{'Method': n, **compute_metrics(y_true_test, p)} for n, p in methods.items()]
metrics_df = pd.DataFrame(metrics_results)
print(metrics_df.to_string(index=False))

In [ ]:
# === Cell 12: 分桶评估 ===
print("="*70)
print("Step 7: 分桶评估")
print("="*70)
for name, y_pred in methods.items():
    print(f"\n【{name}】")
    print(compute_bucket_metrics(y_true_test, y_pred).to_string(index=False))

In [ ]:
# === Cell 13: Bootstrap检验 ===
print("="*70)
print("Step 8: Block Bootstrap 检验")
print("="*70)
comparisons = [('Base+Kalman vs Base', y_pred_test_base, y_pred_test_base_kalman), ('FT+Kalman vs FT', y_pred_test_ft, y_pred_test_ft_kalman), ('Base+AR(1) vs Base', y_pred_test_base, y_pred_test_base_ar1), ('Kalman vs AR(1)', y_pred_test_base_ar1, y_pred_test_base_kalman)]
bootstrap_results = []
for bs in [8, 12, 16]:
    print(f"\n【Block Size = {bs}】")
    for name, pa, pb in comparisons:
        r = block_bootstrap_test(y_true_test, pa, pb, block_size=bs)
        print(f"  {name}: ΔR²={r['delta_r2_mean']:+.4f} [{r['ci_low']:+.4f},{r['ci_high']:+.4f}] win={r['win_rate']:.2f}")
        bootstrap_results.append({'Comparison': name, 'Block_Size': bs, **r})
bootstrap_df = pd.DataFrame(bootstrap_results)

In [ ]:
# === Cell 14: 残差分析 ===
print("="*70)
print("Step 9: 残差时间结构")
print("="*70)
residual_analysis = []
for name, y_pred in methods.items():
    res = y_true_test - y_pred
    acf = compute_acf(res)
    runs = compute_run_stats(res)
    residual_analysis.append({'Method': name, 'Lag1_ACF': acf[1], 'Mean_Run': runs['mean_run_length']})
residual_df = pd.DataFrame(residual_analysis)
print(residual_df.to_string(index=False))

In [ ]:
# === Cell 15: 可视化 ===
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
ax1 = axes[0, 0]
ax1.plot(y_true_test, 'k-', label='y_true', lw=1.5)
ax1.plot(y_pred_test_base, 'b--', label='Base', alpha=0.7)
ax1.plot(y_pred_test_base_kalman, 'r-', label='Base+Kalman', alpha=0.7)
ax1.legend(); ax1.set_title('时序对比'); ax1.grid(True, alpha=0.3)

ax2 = axes[0, 1]
ax2.plot(b_test_base, 'r-', label='Base b_t')
ax2.plot(b_test_ft, 'g-', label='FT b_t')
ax2.axhline(0, color='k', ls='--', lw=0.5)
ax2.legend(); ax2.set_title('偏置漂移轨迹'); ax2.grid(True, alpha=0.3)

ax3 = axes[1, 0]
colors = ['steelblue', 'coral', 'seagreen', 'gold', 'mediumpurple']
bars = ax3.bar(metrics_df['Method'], metrics_df['R2'], color=colors)
ax3.set_ylabel('R²'); ax3.set_title('R² 对比')
for bar, val in zip(bars, metrics_df['R2']):
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002, f'{val:.4f}', ha='center', fontsize=8)
plt.setp(ax3.xaxis.get_majorticklabels(), rotation=15, ha='right')

ax4 = axes[1, 1]
for name, y_pred in [('Base', y_pred_test_base), ('Base+Kalman', y_pred_test_base_kalman), ('Base+AR(1)', y_pred_test_base_ar1)]:
    acf = compute_acf(y_true_test - y_pred)
    ax4.plot(range(11), acf, 'o-', label=name, ms=4)
ax4.axhline(0, color='k', lw=0.5); ax4.legend(); ax4.set_title('残差ACF'); ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('HFENN_kalman_exp_results.png', dpi=150)
plt.show()
print("✅ 图片已保存")

In [ ]:
# === Cell 16: 保存结果 ===
metrics_df.to_csv('HFENN_kalman_metrics.csv', index=False)
bootstrap_df.to_csv('HFENN_kalman_bootstrap.csv', index=False)
residual_df.to_csv('HFENN_kalman_residual.csv', index=False)
pd.DataFrame({'Model': ['Base', 'FT'], 'Q0': [Q0_base, Q0_ft], 'R0': [R0_base, R0_ft], 'Best_Q': [best_Q_base, best_Q_ft], 'Best_R': [best_R_base, best_R_ft], 'AR1_phi': [ar1_base.phi, 0]}).to_csv('HFENN_kalman_params.csv', index=False)
print("✅ 所有结果已保存")

In [ ]:
# === Cell 17: 总结 ===
print("="*70)
print("Bias Drift Kalman 实验总结")
print("="*70)
print(f"\n校准段: {len(calib_idx)} 窗口, 测试段: {len(test_idx)} 窗口")
print(f"\nKalman参数: Base Q={best_Q_base:.6f}, R={best_R_base:.6f}")
print(f"           FT   Q={best_Q_ft:.6f}, R={best_R_ft:.6f}")
print(f"\n性能对比:")
print(metrics_df.to_string(index=False))
base_r2 = metrics_df[metrics_df['Method']=='Base']['R2'].values[0]
kalman_r2 = metrics_df[metrics_df['Method']=='Base+Kalman']['R2'].values[0]
ar1_r2 = metrics_df[metrics_df['Method']=='Base+AR(1)']['R2'].values[0]
print(f"\n结论:")
print(f"  Base+Kalman vs Base: ΔR²={kalman_r2-base_r2:+.4f}")
print(f"  Base+Kalman vs AR(1): ΔR²={kalman_r2-ar1_r2:+.4f}")